# Import all libraries

In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# load Dataset

In [32]:
df = pd.read_csv("../00-dataSource/clean_data_v2.csv")
# Preview data
df.head()

,age,education_level,device_hours_per_day,phone_unlocks,notifications_per_day,social_media_mins,study_mins,physical_activity_days,sleep_hours,sleep_quality,stress_level,productivity_score
0,40,high school,3.54,45,561,98,34,7.0,9.123800,3.353627,6.593289,70.000000
1,27,master,5.65,100,393,174,102,2.0,8.837517,2.908147,4.126926,64.000000
2,31,bachelor,8.87,181,231,595,140,1.0,6.486743,2.889213,1.429139,65.299301
3,41,master,4.05,94,268,18,121,4.0,7.600504,3.097488,4.995512,80.000000
4,26,bachelor,13.07,199,91,147,60,1.0,5.197962,2.786098,9.448757,65.299301


## 4. Feature Engineeing
Feature engineering is the process of transforming and preparing raw data into a format suitable for machine learning models.
This step improves the quality of the input data by:

- Create New Features
- Encode Categorical Features
- Split Features and Target
- Feature Scaling

### 4.1 Creating New Features

In this step, new features are created to better represent user behavior and improve the model’s ability to learn meaningful patterns.

Raw features sometimes do not fully capture the real-world meaning of user habits. Therefore, combining or transforming existing variables can provide stronger and more informative signals.

**Social Media Ratio**:  
  This feature measures the proportion of time spent on social media relative to total device usage.  
  It provides a more accurate representation of user behavior by distinguishing between general device usage and social media engagement.

In [33]:
# 1. Social Media Ratio
df['social_media_ratio'] = df['social_media_mins'] / (df['device_hours_per_day'] * 60 + 1e-5)

**Sleep Efficiency**:  
  This feature combines sleep duration and sleep quality to reflect the overall effectiveness of sleep.  
  Considering both factors together gives a more meaningful indicator of rest and recovery compared to using each feature separately.

In [34]:
# 2. Sleep Efficiency
df['sleep_efficiency'] = df['sleep_hours'] * df['sleep_quality']

**Age Group**:  
  This feature categorizes users into life stages (young, adult, older).  
  Grouping age helps capture differences in lifestyle, behavior, and stress patterns that may not be clearly represented by raw age values.

In [35]:
# 3. Age Group
def age_group(age):
    if age <= 25:
        return 'young'
    elif age <= 40:
        return 'adult'
    else:
        return 'older'

df['age_group'] = df['age'].apply(age_group)

In [36]:
# Preview new features
df[['social_media_ratio', 'sleep_efficiency', 'age_group']].head()

,social_media_ratio,sleep_efficiency,age_group
0,0.461394,30.597824,adult
1,0.513274,25.700797,adult
2,1.118001,18.741587,adult
3,0.074074,23.542471,older
4,0.187452,14.482034,adult


### 4.2 Encoding Categorical Feature

The dataset contains one categorical feature, education level, which represents different levels of educational background.

This feature was encoded into numerical format using one-hot encoding to allow it to be used in machine learning models.

The first category was dropped to avoid redundancy and multicollinearity.

In [37]:
df = pd.get_dummies(df, columns=['education_level', 'age_group'], drop_first=True)

### 4.3 Splitting Features and Target

In this step, the dataset is divided into input features (X) and target variables (y).

The input features (X) are used by the model to learn patterns, while the target variables (y) represent the values the model aims to predict.

This separation is necessary for supervised learning.

In [38]:
# Features (input)
X = df.drop(columns=['stress_level', 'productivity_score'])

# Targets (output)
y_stress = df['stress_level']
y_productivity = df['productivity_score']

***save data*** : the data need to be saved before scaling, because scaling data is unreadable

In [39]:
df.to_csv("../00-dataSource/feature_engineered_data.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


### 4.4 Feature Scaling

Feature scaling is applied to normalize numerical features so that they are on a similar scale.

This is important because machine learning models are sensitive to differences in feature magnitude. Scaling helps improve model performance and ensures that no single feature dominates due to its larger range.

In [40]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)